In [4]:
#Homewokkk

import cv2
import os

disk = "/home/ngoc/drone_ws/src/aruco_detection/homework"

if not os.path.exists(disk):
    os.makedirs(disk)
    print(f"Folder created: {disk}")

cap = cv2.VideoCapture(2) 
img_count = 1
max_images = 40

while True:
    ret, photo = cap.read()
    if not ret:
        print("Failed to read frame from camera")
        break

    cv2.imshow("Camera", photo)

    key = cv2.waitKey(1) & 0xFF

    if key == ord('s'):
        img_name = os.path.join(disk, f"{img_count:02d}.jpg")
        cv2.imwrite(img_name, photo)
        print(f"[{img_count}/{max_images}] successfully save: {img_name}")
        
        img_count += 1
        
        if img_count > max_images:
            print("\nFinished.")
            break
            
    elif key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

[ WARN:0@286.680] global ./modules/videoio/src/cap_gstreamer.cpp (1405) open OpenCV | GStreamer warning: Cannot query video position: status=0, value=-1, duration=-1


[1/40] successfully save: /home/ngoc/drone_ws/src/aruco_detection/homework/01.jpg
[2/40] successfully save: /home/ngoc/drone_ws/src/aruco_detection/homework/02.jpg
[3/40] successfully save: /home/ngoc/drone_ws/src/aruco_detection/homework/03.jpg
[4/40] successfully save: /home/ngoc/drone_ws/src/aruco_detection/homework/04.jpg
[5/40] successfully save: /home/ngoc/drone_ws/src/aruco_detection/homework/05.jpg
[6/40] successfully save: /home/ngoc/drone_ws/src/aruco_detection/homework/06.jpg
[7/40] successfully save: /home/ngoc/drone_ws/src/aruco_detection/homework/07.jpg
[8/40] successfully save: /home/ngoc/drone_ws/src/aruco_detection/homework/08.jpg
[9/40] successfully save: /home/ngoc/drone_ws/src/aruco_detection/homework/09.jpg
[10/40] successfully save: /home/ngoc/drone_ws/src/aruco_detection/homework/10.jpg
[11/40] successfully save: /home/ngoc/drone_ws/src/aruco_detection/homework/11.jpg
[12/40] successfully save: /home/ngoc/drone_ws/src/aruco_detection/homework/12.jpg
[13/40] succe

In [9]:
# Import required libraries for camera calibration and visualization
import cv2, json  # OpenCV for computer vision, json for data handling
import numpy as np  # Numerical operations
import matplotlib.pyplot as plt  # Plotting images
np.set_printoptions(precision=2, suppress=True)  # Format numpy output for readability

# Load the first calibration image from the left camera 
frame_photos = list(range(1,7))
for i in frame_photos:
    filepath = f"aruco_detection/homework/{i:02}.jpg"
    img = cv2.imread(filepath)

img = cv2.imread(f"aruco_detection/homework/{i:02}.jpg")

plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))        
# plt.show()


h, w = img.shape[:2] #chỉ lấy 2 giá trị đầu tiên (chiều cao và chiều rộng), bỏ qua số lượng kênh màu (số 3).

# Define the shape of the chessboard (number of inner corners per row and column)
board_shape = (8, 6)  # 9 columns and 6 rows of inner corners

# Size of one checker square in millimeters
checker_size = 35  # in mm


criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

objp = np.zeros((board_shape[1]*board_shape[0], 3), np.float32) # (54,3) . . . 
objp[:, :2] =  np.mgrid[0:board_shape[0], 0: board_shape[1]].T.reshape(-1, 2)

objp *= checker_size  # scale to mm
objpoints = []  # 3D points in board coords 
img_shape = (w, h)

frame_ids = list(range(0, 2, 1))  # Adjust as needed for more images
for i in frame_ids:
    # Load the current image
    frame = cv2.imread(f"aruco_detection/homework/{i:02}.jpg")
    
    # Find chessboard corners in the image
    ret, corners = cv2.findChessboardCorners(frame, board_shape, flags=cv2.CALIB_CB_ADAPTIVE_THRESH + cv2.CALIB_CB_NORMALIZE_IMAGE)
    
    # Refine corner locations to subpixel accuracy
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    corners_ref = cv2.cornerSubPix(gray_frame, corners, (11, 11), (-1, -1), criteria)
    
    # Draw the refined corners on the image for visualization 
    frame = cv2.drawChessboardCorners(frame, board_shape, corners_ref, ret)
    

# Loop through a small set of calibration images to visualize corner detection
frame_ids = list(range(0, 10, 1))  # Adjust as needed for more images
for i in frame_ids:
    # Load the current image
    frame = cv2.imread(f"aruco_detection/homework/{i:02}.jpg")
    
    # Find chessboard corners in the image
    ret, corners = cv2.findChessboardCorners(frame, board_shape, flags=cv2.CALIB_CB_ADAPTIVE_THRESH + cv2.CALIB_CB_NORMALIZE_IMAGE)
    
    # Refine corner locations to subpixel accuracy
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    corners_ref = cv2.cornerSubPix(gray_frame, corners, (11, 11), (-1, -1), criteria)
    
    # Draw the refined corners on the image for visualization 
    frame = cv2.drawChessboardCorners(frame, board_shape, corners_ref, ret)


    # Store the object points and image points for calibration
    objpoints.append(objp.copy())
    imgpoints.append(corners_ref)
    
# Set calibration flags
flags = cv2.CALIB_FIX_K3 | cv2.CALIB_USE_INTRINSIC_GUESS  # Fix k3 distortion, use initial guess for intrinsics

# Initial camera matrix setup (intrinsic parameters)
initial_fx = 1000.0  # Initial horizontal focal length
initial_fy = 1000.0  # Initial vertical focal length (square pixels)
image_width = 640
image_height = 480
initial_cx = image_width / 2  # Principal point x
initial_cy = image_height / 2  # Principal point y
initial_camera_matrix = np.array([[initial_fx, 0, initial_cx],
                                [0, initial_fy, initial_cy],
                                [0, 0, 1]], dtype=np.float32)

# Run camera calibration using collected object and image points
ret, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(
        objpoints, imgpoints, img_shape, 
        cameraMatrix=initial_camera_matrix, distCoeffs=None,
        flags=flags, 
        criteria=criteria
    )

# Print calibration results
print("\n=== Calibration Results (LEFT) ===")
print(f"RMS reprojection error: {ret:.4f}")
print("K (intrinsics):\n", mtx)
print("dist (k1 k2 p1 p2 k3):\n", dist.ravel())

# Undistort the image using the computed camera matrix and distortion coefficients
undistorted = cv2.undistort(frame, mtx, dist)

# Visualize the original and undistorted images side by side
plt.figure(figsize=(16, 9))
plt.subplot(1, 2, 1)
plt.axis('off')
plt.title("Original")
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.subplot(1, 2, 2)
plt.axis('off')
plt.title("Undistorted")
plt.imshow(cv2.cvtColor(undistorted, cv2.COLOR_BGR2RGB))
plt.show()

[ WARN:0@1444.871] global ./modules/imgcodecs/src/loadsave.cpp (239) findDecoder imread_('aruco_detection/homework/01.jpg'): can't open/read file: check file path/integrity
[ WARN:0@1444.872] global ./modules/imgcodecs/src/loadsave.cpp (239) findDecoder imread_('aruco_detection/homework/02.jpg'): can't open/read file: check file path/integrity
[ WARN:0@1444.872] global ./modules/imgcodecs/src/loadsave.cpp (239) findDecoder imread_('aruco_detection/homework/03.jpg'): can't open/read file: check file path/integrity
[ WARN:0@1444.872] global ./modules/imgcodecs/src/loadsave.cpp (239) findDecoder imread_('aruco_detection/homework/04.jpg'): can't open/read file: check file path/integrity
[ WARN:0@1444.872] global ./modules/imgcodecs/src/loadsave.cpp (239) findDecoder imread_('aruco_detection/homework/05.jpg'): can't open/read file: check file path/integrity
[ WARN:0@1444.872] global ./modules/imgcodecs/src/loadsave.cpp (239) findDecoder imread_('aruco_detection/homework/06.jpg'): can't open

error: OpenCV(4.6.0) ./modules/imgproc/src/color.cpp:182: error: (-215:Assertion failed) !_src.empty() in function 'cvtColor'
